# 3.4: Initializing residual branches

Large language models combine both habits: pre-norm (or RMSNorm) residual blocks, and scaled-down initialization of the projections that write back into the stream.

A simple illustration for an MLP residual stack:

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.

# From The residual block
import torch

import torch.nn as nn


In [ ]:
class ScaledResidualBlock(nn.Module):
    def __init__(self, width, depth_hint):
        super().__init__()
        self.f = nn.Sequential(
            nn.Linear(width, width),
            nn.ReLU(),
            nn.Linear(width, width),
        )
        nn.init.kaiming_normal_(self.f[0].weight, nonlinearity="relu")
        nn.init.zeros_(self.f[0].bias)
        # Scale the last layer down so L blocks add O(1) variance at init.
        nn.init.kaiming_normal_(self.f[-1].weight, nonlinearity="linear")
        self.f[-1].weight.data *= depth_hint ** -0.5
        nn.init.zeros_(self.f[-1].bias)

    def forward(self, x):
        return x + self.f(x)


Exact scaling conventions differ across codebases; the principle is to keep the residual stream from exploding as depth increases *at initialization*, then let learning grow the updates as needed. You can check the numbers quoted above by stacking these blocks with and without the scaling line and printing the standard deviation of the stream on random inputs:


In [ ]:
torch.manual_seed(0)
for L in [4, 16, 64]:
    x = torch.randn(512, 256)
    blocks = [ScaledResidualBlock(256, depth_hint=L) for _ in range(L)]
    with torch.no_grad():
        for b in blocks:
            x = b(x)
    print(L, round(x.std().item(), 2))   # stays near 1.5-1.7 for every depth
